In [1]:
import collections
import gymnasium as gym
import numpy as np
import pandas as pd

env = gym.make("Blackjack-v1", natural=False, sab=False)


def run_episode(env, policy_fn):
    """Generates a complete episode using a given policy function."""
    episode, state, done = [], env.reset()[0], False
    while not done:
        action = policy_fn(state)
        next_state, reward, term, trunc, _ = env.step(action)
        done = term or trunc
        episode.append((state, action, reward))
        state = next_state
    return episode


# ==========================================
# 1. FIRST-VISIT MC PREDICTION
# ==========================================
def mc_prediction(env, num_episodes=100000):
    policy = lambda s: 0 if s[0] >= 20 else 1  # Stick if >= 20, else Hit
    V, r_sum, r_count = (
        collections.defaultdict(float),
        collections.defaultdict(float),
        collections.defaultdict(int),
    )

    for _ in range(num_episodes):
        episode = run_episode(env, policy)
        visited, G = set(), 0
        for state, _, reward in reversed(episode):
            G += reward
            if state not in visited:
                visited.add(state)
                r_sum[state] += G
                r_count[state] += 1
                V[state] = r_sum[state] / r_count[state]
    return V


# ==========================================
# 2. ON-POLICY MC CONTROL (Epsilon-Greedy)
# ==========================================
def mc_control(env, num_episodes=50000, eps=0.15):
    Q = collections.defaultdict(lambda: np.zeros(2))
    r_sum, r_count = collections.defaultdict(float), collections.defaultdict(
        int
    )

    for _ in range(num_episodes):
        # Epsilon-greedy selection
        def policy_fn(s):
            if np.random.rand() < eps:
                return env.action_space.sample()
            return np.argmax(Q[s])

        episode = run_episode(env, policy_fn)
        visited, G = set(), 0
        for state, action, reward in reversed(episode):
            G += reward
            sa_pair = (state, action)
            if sa_pair not in visited:
                visited.add(sa_pair)
                r_sum[sa_pair] += G
                r_count[sa_pair] += 1
                Q[state][action] = r_sum[sa_pair] / r_count[sa_pair]
    return Q


# --- Execution ---
print("Running Prediction & Control simulations...")
V_table = mc_prediction(env, num_episodes=100000)
Q_table = mc_control(env, num_episodes=50000)

# Display a quick slice of the Control policy
df = pd.DataFrame(
    [
        {
            "State (Player, Dealer, Ace)": s,
            "Stick Value": act[0],
            "Hit Value": act[1],
            "Best Action": "Stick" if act[0] > act[1] else "Hit",
        }
        for s, act in Q_table.items()
    ]
)
print("\nSample of learned optimal decisions:")
print(df.head(5))


Running Prediction & Control simulations...

Sample of learned optimal decisions:
  State (Player, Dealer, Ace)  Stick Value  Hit Value Best Action
0                 (10, 10, 0)    -0.534247  -0.106209         Hit
1                  (20, 7, 0)     0.739623  -0.966667       Stick
2                  (17, 9, 0)    -0.413793  -0.793103       Stick
3                  (16, 2, 0)    -0.271233  -0.393443       Stick
4                  (20, 1, 0)     0.113290  -0.750000       Stick
